In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Admin\Downloads\movie prediction.csv", encoding='latin1')


# Drop rows with missing ratings
df = df.dropna(subset=['Rating']).reset_index(drop=True)

# -----------------------
# Data Cleaning
# -----------------------
df['Year_clean'] = df['Year'].astype(str).str.extract(r'(\d{4})').astype(float)
df['Duration_min'] = df['Duration'].astype(str).str.extract(r'(\d+)').astype(float)

# Clean votes
def clean_votes(x):
    try:
        return int(str(x).replace(',', '').strip())
    except:
        return np.nan
df['Votes_num'] = df['Votes'].apply(clean_votes)

# Fill missing for categorical
for col in ['Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3']:
    df[col] = df[col].fillna('Unknown').astype(str)

# -----------------------
# Genre Analysis
# -----------------------
df['Genre_list'] = df['Genre'].str.replace('|', ',', regex=False).str.replace('/', ',', regex=False)
df['num_genres'] = df['Genre_list'].apply(lambda x: len([g for g in [i.strip() for i in x.split(',')] if g and g.lower() != 'unknown']))

# Count top genres
genre_counts = df['Genre_list'].str.split(',', expand=True).stack().str.strip().value_counts()
top_genres = genre_counts.head(20)

# -----------------------
# Director Analysis
# -----------------------
top_directors = df['Director'].value_counts().head(10)

# -----------------------
# Actor Analysis
# -----------------------
all_actors = pd.concat([df['Actor 1'], df['Actor 2'], df['Actor 3']])
top_actors = all_actors.value_counts().head(20)

# -----------------------
# Save cleaned data
# -----------------------
df.to_csv("cleaned_movie_data.csv", index=False)

# -----------------------
# Display summaries
# -----------------------
print("\nDataset shape after cleaning:", df.shape)
print("\nTop 20 Genres:\n", top_genres)
print("\nTop 10 Directors:\n", top_directors)
print("\nTop 20 Actors:\n", top_actors)
